## 02 — Modelagem da propensão de compra

**Projeto:** Propensão de compra em e-commerce  
**Pós-Tech:** Machine Learning Engineering — Fase 2  
**Grupo:** 24

### Objetivo

Construir e comparar modelos de classificação para estimar a propensão de compra com base nas características disponíveis no dataset.

O desenvolvimento contempla:

- Definição do cenário de previsão e das características utilizadas.
- Pré-processamento integrado ao modelo.
- Construção de uma referência simples.
- Comparação entre Regressão Logística e Random Forest.
- Registro de experimentos no MLflow.
- Seleção da configuração utilizando treinamento e validação.
- Avaliação final no conjunto de teste.


### Cuidados metodológicos
As transformações serão ajustadas exclusivamente nos dados de treinamento de cada ajuste.

O conjunto de validação será utilizado para comparar configurações e selecionar o limiar de decisão.

O conjunto de teste será reservado para a avaliação final, após a definição da configuração escolhida.

A utilização de características depende de sua construção e disponibilidade no momento da previsão. As decisões de inclusão e exclusão serão documentadas antes do treinamento. A disponibilidade das demais características em uma aplicação real permanece pendente de verificação.

Os registros com características idênticas serão preservados conforme a decisão registrada na análise exploratória. A sobreposição entre conjuntos permanece como uma limitação.

---

In [4]:
# ============================================================
# CONFIGURAÇÃO DO AMBIENTE E IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# CONFIGURAÇÕES DE VISUALIZAÇÃO E REPRODUTIBILIDADE
# ============================================================

RANDOM_STATE = 42

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")


# ============================================================
# LOCALIZAÇÃO DA RAIZ DO PROJETO
# ============================================================

current_directory = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (
            current_directory,
            *current_directory.parents,
        )
        if (directory / "pyproject.toml").is_file()
        and (directory / "data").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Raiz do projeto não encontrada. "
        "Execute o notebook dentro da pasta do projeto."
    )


# ============================================================
# LOCALIZAÇÃO DOS DADOS E DEFINIÇÃO DA VARIÁVEL-ALVO
# ============================================================

PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"

TARGET_COLUMN = "Revenue"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretório dos dados: {PROCESSED_DATA_DIRECTORY}")
print(f"Semente para os modelos: {RANDOM_STATE}")

Raiz do projeto: /Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops
Diretório dos dados: /Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops/data/processed
Semente para os modelos: 42


---

### 1. Cenário de modelagem e disponibilidade das informações

O experimento utilizará as características agregadas de navegação disponíveis no dataset para estimar a ocorrência de compra.

Como os dados utilizados não fornecem recortes intermediários da navegação, esta avaliação não demonstra capacidade de antecipar compras em um instante específico da sessão.

Uma aplicação durante a navegação exigiria características calculadas apenas com as informações disponíveis até o momento da previsão e uma avaliação correspondente.


#### Decisão inicial sobre o valor das páginas
A variável PageValues será excluída dos experimentos iniciais, pois sua construção e disponibilidade no momento da previsão ainda não foram verificadas.

Essa exclusão é uma decisão conservadora de escopo, não uma comprovação de vazamento de informação.

As demais características também deverão ter sua disponibilidade verificada antes de uma aplicação real.


#### Separação dos conjuntos
Serão utilizados os conjuntos de treinamento e validação já produzidos pelo pipeline do projeto.

O conjunto de teste não será carregado nesta etapa. Nenhuma nova divisão será realizada neste notebook.

In [5]:
# ============================================================
# LOCALIZAÇÃO DOS CONJUNTOS DE TREINAMENTO E VALIDAÇÃO
# ============================================================

TRAIN_DATASET_PATH = (
    PROCESSED_DATA_DIRECTORY / "train.parquet"
)

VALIDATION_DATASET_PATH = (
    PROCESSED_DATA_DIRECTORY / "validation.parquet"
)

for dataset_path in (
    TRAIN_DATASET_PATH,
    VALIDATION_DATASET_PATH,
):
    if not dataset_path.is_file():
        raise FileNotFoundError(
            f"Conjunto de dados não encontrado: {dataset_path}"
        )


# ============================================================
# CARREGAMENTO DOS CONJUNTOS
# ============================================================

df_train = pd.read_parquet(TRAIN_DATASET_PATH)
df_validation = pd.read_parquet(VALIDATION_DATASET_PATH)


# ============================================================
# VERIFICAÇÃO DA ESTRUTURA E DA VARIÁVEL-ALVO
# ============================================================

if set(df_train.columns) != set(df_validation.columns):
    raise ValueError(
        "Treinamento e validação possuem colunas diferentes."
    )

for dataset_name, dataset in (
    ("Treinamento", df_train),
    ("Validação", df_validation),
):
    if dataset.empty:
        raise ValueError(
            f"O conjunto de {dataset_name.lower()} está vazio."
        )

    if TARGET_COLUMN not in dataset.columns:
        raise ValueError(
            f"A variável-alvo '{TARGET_COLUMN}' não está "
            f"no conjunto de {dataset_name.lower()}."
        )

    if dataset[TARGET_COLUMN].isna().any():
        raise ValueError(
            f"Existem valores ausentes no alvo de "
            f"{dataset_name.lower()}."
        )

    if not dataset[TARGET_COLUMN].isin([False, True]).all():
        raise ValueError(
            f"O alvo de {dataset_name.lower()} deve conter "
            "somente valores booleanos ou códigos 0 e 1."
        )

    if dataset[TARGET_COLUMN].nunique() != 2:
        raise ValueError(
            f"O conjunto de {dataset_name.lower()} "
            "deve conter as duas classes."
        )


# ============================================================
# RESUMO DOS CONJUNTOS CARREGADOS
# ============================================================

dataset_summary = pd.DataFrame(
    {
        "Conjunto": ["Treinamento", "Validação"],
        "Quantidade de sessões": [
            len(df_train),
            len(df_validation),
        ],
        "Quantidade de colunas": [
            df_train.shape[1],
            df_validation.shape[1],
        ],
    }
)

display(dataset_summary)

print("Conjuntos carregados e estrutura verificada.")
print("O conjunto de teste permanece reservado.")

,Conjunto,Quantidade de sessões,Quantidade de colunas
0,Treinamento,8631,18
1,Validação,1849,18


Conjuntos carregados e estrutura verificada.
O conjunto de teste permanece reservado.


---

### 2. Definição das características e da variável-alvo

A variável-alvo será Revenue, representada por:

- 0: sessão sem compra.
- 1: sessão com compra.

As características serão separadas em numéricas e categóricas.

Os códigos de sistema operacional, navegador, região e tipo de tráfego serão tratados como categorias, sem atribuir ordem ou distância entre seus valores.

PageValues será excluída conforme a decisão inicial de escopo. Revenue não fará parte das características de entrada.

Nesta etapa, não serão ajustadas codificações, escalonamentos ou outras transformações.

In [6]:
# ============================================================
# DEFINIÇÃO DAS VARIÁVEIS NUMÉRICAS
# ============================================================

NUMERIC_COLUMNS = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "SpecialDay",
]


# ============================================================
# DEFINIÇÃO DAS VARIÁVEIS CATEGÓRICAS
# ============================================================

CATEGORICAL_COLUMNS = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

FEATURE_COLUMNS = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS

EXCLUDED_COLUMNS = ["PageValues"]


# ============================================================
# VERIFICAÇÃO DAS COLUNAS DEFINIDAS
# ============================================================

if len(FEATURE_COLUMNS) != len(set(FEATURE_COLUMNS)):
    raise ValueError(
        "Existem características repetidas na configuração."
    )

if TARGET_COLUMN in FEATURE_COLUMNS:
    raise ValueError(
        "A variável-alvo não pode integrar as características."
    )

expected_columns = set(
    FEATURE_COLUMNS + EXCLUDED_COLUMNS + [TARGET_COLUMN]
)

for dataset_name, dataset in (
    ("Treinamento", df_train),
    ("Validação", df_validation),
):
    missing_columns = expected_columns - set(dataset.columns)
    unexpected_columns = set(dataset.columns) - expected_columns

    if missing_columns or unexpected_columns:
        raise ValueError(
            f"Colunas incompatíveis em {dataset_name}: "
            f"ausentes={sorted(missing_columns)}; "
            f"não previstas={sorted(unexpected_columns)}."
        )


# ============================================================
# SEPARAÇÃO DAS CARACTERÍSTICAS E DO ALVO
# ============================================================

X_train = df_train[FEATURE_COLUMNS].copy()
y_train = df_train[TARGET_COLUMN].astype(int).copy()

X_validation = df_validation[FEATURE_COLUMNS].copy()
y_validation = df_validation[TARGET_COLUMN].astype(int).copy()


# ============================================================
# RESUMO DAS ENTRADAS PARA A MODELAGEM
# ============================================================

modeling_summary = pd.DataFrame(
    {
        "Conjunto": ["Treinamento", "Validação"],
        "Quantidade de sessões": [
            len(X_train),
            len(X_validation),
        ],
        "Variáveis numéricas": [
            len(NUMERIC_COLUMNS),
            len(NUMERIC_COLUMNS),
        ],
        "Variáveis categóricas": [
            len(CATEGORICAL_COLUMNS),
            len(CATEGORICAL_COLUMNS),
        ],
        "Total de características": [
            X_train.shape[1],
            X_validation.shape[1],
        ],
    }
)

display(modeling_summary)

print("Variável-alvo: compra realizada (0 = não; 1 = sim).")
print("Característica excluída: PageValues.")
print("Nenhuma transformação foi ajustada nesta etapa.")

,Conjunto,Quantidade de sessões,Variáveis numéricas,Variáveis categóricas,Total de características
0,Treinamento,8631,9,7,16
1,Validação,1849,9,7,16


Variável-alvo: compra realizada (0 = não; 1 = sim).
Característica excluída: PageValues.
Nenhuma transformação foi ajustada nesta etapa.


#### Resultado da preparação das entradas

O treinamento contém 8.631 sessões, e a validação contém 1.849 sessões.

Ambos possuem 16 características de entrada: nove numéricas e sete categóricas, organizadas na mesma ordem.

A variável-alvo foi separada das características e codificada como 0 para ausência de compra e 1 para compra.

PageValues foi excluída conforme a decisão inicial de escopo. Nenhuma codificação, transformação ou normalização foi ajustada nesta etapa.

O conjunto de teste permanece reservado para a avaliação final.

---

### 3. Pré-processamento das características

As variáveis categóricas serão codificadas com One-Hot Encoding, sem atribuir ordem aos códigos numéricos.

Categorias não observadas no treinamento serão representadas por zeros nas colunas correspondentes àquela variável. Isso permite processar novas categorias, mas não significa que o modelo tenha aprendido seu comportamento.

Para a Regressão Logística, as variáveis numéricas serão padronizadas com StandardScaler.

Para a Random Forest, as variáveis numéricas permanecerão na escala original.

Neste primeiro experimento, não serão aplicadas transformações logarítmicas, reamostragem ou remoção de valores extremos.

O pré-processamento será integrado aos modelos por meio de pipelines. Seus parâmetros serão aprendidos exclusivamente no treinamento de cada ajuste e reutilizados na validação.

In [7]:
# ============================================================
# IMPORTAÇÃO DOS COMPONENTES DE PRÉ-PROCESSAMENTO
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ============================================================
# PRÉ-PROCESSAMENTO PARA A REGRESSÃO LOGÍSTICA
# ============================================================

logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_COLUMNS,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_COLUMNS,
        ),
    ],
    remainder="drop",
)


# ============================================================
# PRÉ-PROCESSAMENTO PARA A RANDOM FOREST
# ============================================================

forest_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            NUMERIC_COLUMNS,
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_COLUMNS,
        ),
    ],
    remainder="drop",
)


# ============================================================
# RESUMO DAS ESTRATÉGIAS
# ============================================================

preprocessing_summary = pd.DataFrame(
    {
        "Modelo": [
            "Regressão Logística",
            "Random Forest",
        ],
        "Variáveis numéricas": [
            "Padronização com StandardScaler",
            "Escala original",
        ],
        "Variáveis categóricas": [
            "One-Hot Encoding",
            "One-Hot Encoding",
        ],
        "Categorias desconhecidas": [
            "Ignoradas pelo codificador",
            "Ignoradas pelo codificador",
        ],
    }
)

display(preprocessing_summary)

print("Pré-processadores configurados.")
print("Nenhum ajuste foi realizado nos dados.")

,Modelo,Variáveis numéricas,Variáveis categóricas,Categorias desconhecidas
0,Regressão Logística,Padronização com StandardScaler,One-Hot Encoding,Ignoradas pelo codificador
1,Random Forest,Escala original,One-Hot Encoding,Ignoradas pelo codificador


Pré-processadores configurados.
Nenhum ajuste foi realizado nos dados.


#### Resultado da configuração do pré-processamento

Foram configuradas duas estratégias de pré-processamento, uma para cada modelo.

Na Regressão Logística, as nove variáveis numéricas serão padronizadas com StandardScaler. Na Random Forest, essas variáveis permanecerão na escala original.

As sete variáveis categóricas serão codificadas com One-Hot Encoding nos dois modelos, sem atribuir ordem aos códigos das categorias.

Categorias não observadas durante o treinamento serão representadas por zeros nas colunas da respectiva variável.

Nesta etapa, apenas os pré-processadores foram configurados. Nenhuma transformação foi ajustada ou aplicada aos dados.

O ajuste ocorrerá dentro do pipeline de cada modelo, exclusivamente com os dados de treinamento. A validação utilizará as transformações aprendidas nesse ajuste.

---

### 4. Métricas e critérios de avaliação

Average Precision será a métrica principal para comparar os modelos na validação, considerando as sessões com compra como classe positiva.

Essa métrica resume a relação entre precisão e recall ao variar o limiar de decisão. Não será interpretada como percentual de acertos.

Também serão acompanhadas:

- Precisão: proporção de compras entre as sessões classificadas como compra.
- Recall: proporção das compras identificadas pelo modelo.
- F1: média harmônica entre precisão e recall.
- Acurácia: proporção total de classificações corretas.
- Matriz de confusão: contagens de acertos e erros por classe.

A comparação inicial utilizará limiar de 0,50. O ajuste do limiar será realizado posteriormente na validação, após a seleção do modelo.

Como não foram definidos custos financeiros para os erros, o F1 será utilizado como critério técnico inicial para selecionar o limiar. Esse critério deverá ser revisto conforme os objetivos de uma aplicação real.

O conjunto de teste permanecerá reservado para avaliar a configuração final escolhida.

In [8]:
# ============================================================
# CONFIGURAÇÃO DOS CRITÉRIOS DE AVALIAÇÃO
# ============================================================

PRIMARY_METRIC = "average_precision"
INITIAL_THRESHOLD = 0.50
THRESHOLD_SELECTION_METRIC = "f1"


# ============================================================
# APRESENTAÇÃO DOS CRITÉRIOS
# ============================================================

evaluation_criteria = pd.DataFrame(
    {
        "Decisão": [
            "Comparação dos modelos",
            "Limiar inicial",
            "Seleção posterior do limiar",
            "Classe positiva",
            "Avaliação final",
        ],
        "Critério": [
            "Maior Average Precision na validação",
            f"{INITIAL_THRESHOLD:.2f}".replace(".", ","),
            "Maior F1 na validação",
            "Compra realizada (1)",
            "Teste após a definição do modelo e do limiar",
        ],
    }
)

display(evaluation_criteria)

print("Critérios definidos antes do treinamento.")

,Decisão,Critério
0,Comparação dos modelos,Maior Average Precision na validação
1,Limiar inicial,"0,50"
2,Seleção posterior do limiar,Maior F1 na validação
3,Classe positiva,Compra realizada (1)
4,Avaliação final,Teste após a definição do modelo e do limiar


Critérios definidos antes do treinamento.


#### Resultado da definição dos critérios de avaliação

Average Precision na validação foi definida como a métrica principal para comparar os modelos, considerando compra realizada (1) como classe positiva.

Na avaliação inicial, será utilizado o limiar de 0,50. Precisão, recall, F1, acurácia e matriz de confusão complementarão a análise.

Após a seleção do modelo, o limiar será escolhido pelo maior F1 na validação. Esse critério é técnico e provisório, pois não foram definidos custos de negócio para os erros.

O conjunto de teste será utilizado somente após a definição do modelo e do limiar.

Nenhum modelo foi treinado nesta etapa. Os critérios foram estabelecidos antes da observação dos resultados.

---

### 5. Configuração do rastreamento de experimentos

O MLflow será utilizado para registrar os parâmetros, as métricas e os modelos de cada experimento.

Os metadados serão armazenados em um banco SQLite local. Os artefatos serão armazenados em uma pasta do projeto.

As execuções serão organizadas em um experimento dedicado à modelagem da propensão de compra.

Nesta etapa, será configurado apenas o ambiente de rastreamento. Nenhum modelo será treinado ou registrado no Model Registry.

In [9]:
# ============================================================
# IMPORTAÇÃO DO MLFLOW
# ============================================================

import mlflow


# ============================================================
# CONFIGURAÇÃO DO ARMAZENAMENTO LOCAL
# ============================================================

MLFLOW_DIRECTORY = PROJECT_ROOT / "mlruns"
MLFLOW_ARTIFACT_DIRECTORY = MLFLOW_DIRECTORY / "artifacts"
MLFLOW_DATABASE_PATH = MLFLOW_DIRECTORY / "mlflow.db"

MLFLOW_ARTIFACT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

MLFLOW_TRACKING_URI = (
    f"sqlite:///{MLFLOW_DATABASE_PATH.as_posix()}"
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_TRACKING_URI)


# ============================================================
# CRIAÇÃO OU REUTILIZAÇÃO DO EXPERIMENTO
# ============================================================

EXPERIMENT_NAME = "purchase_propensity_modeling"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=MLFLOW_ARTIFACT_DIRECTORY.as_uri(),
    )

experiment = mlflow.set_experiment(EXPERIMENT_NAME)


# ============================================================
# RESUMO DA CONFIGURAÇÃO
# ============================================================

mlflow_configuration = pd.DataFrame(
    {
        "Configuração": [
            "Versão do MLflow",
            "Experimento",
            "Identificador do experimento",
            "Banco de metadados",
            "Local dos artefatos",
        ],
        "Valor": [
            mlflow.__version__,
            experiment.name,
            experiment.experiment_id,
            str(MLFLOW_DATABASE_PATH),
            experiment.artifact_location,
        ],
    }
)

display(mlflow_configuration)

print("MLflow configurado para os próximos experimentos.")
print("Nenhum treinamento foi realizado nesta etapa.")

/Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/18 15:15:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/18 15:15:45 INFO mlflow.store.db.utils: Updating database tables


,Configuração,Valor
0,Versão do MLflow,3.16.0
1,Experimento,purchase_propensity_modeling
2,Identificador do experimento,1
3,Banco de metadados,/Users/biancafirmino/Developer/ecommerce-purch...
4,Local dos artefatos,file:///Users/biancafirmino/Developer/ecommerc...


MLflow configurado para os próximos experimentos.
Nenhum treinamento foi realizado nesta etapa.


#### Resultado da configuração do MLflow

O MLflow foi configurado com armazenamento local em SQLite para os metadados e uma pasta específica para os artefatos.

O experimento purchase_propensity_modeling foi selecionado com identificador 1, permitindo organizar as próximas execuções de treinamento.

Nenhuma execução de treinamento foi criada nesta etapa. Ainda não foram registrados parâmetros, métricas ou modelos.

A configuração do rastreamento foi concluída. O registro efetivo dos experimentos será verificado nas próximas etapas.

---

### 6. Modelo de referência

Será utilizado um DummyClassifier com estratégia prior.

Essa estratégia aprende somente as proporções das classes no treinamento e atribui as mesmas probabilidades a todas as sessões, independentemente das características.

Com o limiar inicial de 0,50 e a predominância de sessões sem compra, a referência deverá classificar todas as sessões como ausência de compra.

Esse comportamento permite contextualizar a acurácia: um resultado elevado nessa métrica pode coexistir com recall zero para as compras.

O modelo será ajustado no treinamento e avaliado na validação. Seus parâmetros, métricas e artefato serão registrados no MLflow.

O conjunto de teste permanecerá reservado.

In [10]:
# ============================================================
# IMPORTAÇÃO DO MODELO DE REFERÊNCIA E DAS MÉTRICAS
# ============================================================

import mlflow.sklearn

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# ============================================================
# TREINAMENTO E AVALIAÇÃO DO MODELO DE REFERÊNCIA
# ============================================================

with mlflow.start_run(run_name="baseline_prior") as run:
    baseline_model = DummyClassifier(
        strategy="prior",
        random_state=RANDOM_STATE,
    )

    baseline_model.fit(X_train, y_train)

    positive_class_index = list(
        baseline_model.classes_
    ).index(1)

    baseline_probabilities = baseline_model.predict_proba(
        X_validation
    )[:, positive_class_index]

    baseline_predictions = (
        baseline_probabilities >= INITIAL_THRESHOLD
    ).astype(int)

    baseline_metrics = {
        "validation_average_precision": average_precision_score(
            y_validation,
            baseline_probabilities,
        ),
        "validation_accuracy": accuracy_score(
            y_validation,
            baseline_predictions,
        ),
        "validation_precision": precision_score(
            y_validation,
            baseline_predictions,
            zero_division=0,
        ),
        "validation_recall": recall_score(
            y_validation,
            baseline_predictions,
            zero_division=0,
        ),
        "validation_f1": f1_score(
            y_validation,
            baseline_predictions,
            zero_division=0,
        ),
    }

    baseline_confusion_matrix = confusion_matrix(
        y_validation,
        baseline_predictions,
        labels=[0, 1],
    )


    # ========================================================
    # REGISTRO DO EXPERIMENTO NO MLFLOW
    # ========================================================

    mlflow.log_params(
        {
            "model_type": "DummyClassifier",
            "strategy": "prior",
            "random_state": RANDOM_STATE,
            "threshold": INITIAL_THRESHOLD,
            "train_rows": len(X_train),
            "validation_rows": len(X_validation),
            "feature_count": X_train.shape[1],
            "excluded_features": ",".join(EXCLUDED_COLUMNS),
        }
    )

    mlflow.set_tags(
        {
            "role": "baseline",
            "evaluation_dataset": "validation",
        }
    )

    mlflow.log_metrics(baseline_metrics)

    mlflow.log_dict(
        {
            "labels": [0, 1],
            "rows": "actual",
            "columns": "predicted",
            "matrix": baseline_confusion_matrix.tolist(),
        },
        "evaluation/confusion_matrix.json",
    )

    baseline_model_info = mlflow.sklearn.log_model(
        sk_model=baseline_model,
        name="baseline_model",
        input_example=X_train.head(5),
    )

    baseline_run_id = run.info.run_id


# ============================================================
# APRESENTAÇÃO DAS MÉTRICAS EM PORTUGUÊS
# ============================================================

baseline_results = pd.DataFrame(
    {
        "Métrica": [
            "Average Precision",
            "Acurácia",
            "Precisão",
            "Recall",
            "F1",
        ],
        "Valor na validação": [
            baseline_metrics["validation_average_precision"],
            baseline_metrics["validation_accuracy"],
            baseline_metrics["validation_precision"],
            baseline_metrics["validation_recall"],
            baseline_metrics["validation_f1"],
        ],
    }
)

display(baseline_results)


# ============================================================
# APRESENTAÇÃO DA MATRIZ DE CONFUSÃO
# ============================================================

baseline_confusion_display = pd.DataFrame(
    baseline_confusion_matrix,
    index=["Sem compra", "Com compra"],
    columns=["Sem compra", "Com compra"],
).rename_axis(
    index="Resultado real",
    columns="Resultado previsto",
)

display(baseline_confusion_display)

print(f"Identificador da execução: {baseline_run_id}")
print(f"Modelo salvo no MLflow: {baseline_model_info.model_uri}")
print("O conjunto de teste permanece reservado.")

/Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer colum

,Métrica,Valor na validação
0,Average Precision,0.1547
1,Acurácia,0.8453
2,Precisão,0.0000
3,Recall,0.0000
4,F1,0.0000


Resultado previsto,Sem compra,Com compra
Resultado real,,
Sem compra,1563,0
Com compra,286,0


Identificador da execução: 7e57c47558f64a3d9390d8d8ddf32baf
Modelo salvo no MLflow: models:/m-8016618870c645a9bfdd80e129a37d31
O conjunto de teste permanece reservado.


#### Resultados do modelo de referência

O modelo de referência classificou todas as 1.849 sessões da validação como ausência de compra.

A matriz de confusão apresenta:

- 1.563 sessões sem compra classificadas corretamente.
- 286 sessões com compra classificadas como ausência de compra.
- Nenhuma sessão classificada como compra.

A acurácia foi de aproximadamente 84,53%, enquanto recall e F1 foram iguais a zero. A precisão foi apresentada como zero pela convenção definida no código, pois não houve previsões positivas.

Esse resultado demonstra por que a acurácia isolada não é suficiente para avaliar o problema: a referência acerta a classe majoritária, mas não identifica as compras.

Average Precision foi de aproximadamente 0,1547, equivalente à proporção de sessões com compra na validação. Como todas as sessões receberam a mesma probabilidade, o modelo não distingue quais possuem maior propensão de compra.

A execução foi concluída com registro dos parâmetros, métricas, matriz de confusão e modelo no MLflow. O modelo ainda não foi registrado no Model Registry.

Esses resultados serão utilizados como referência para avaliar os modelos que utilizam as características de navegação.

O conjunto de teste permanece reservado.

---

### 7. Regressão Logística

A Regressão Logística será utilizada como primeiro modelo que considera as características de navegação.

O pipeline integrará a padronização das variáveis numéricas, a codificação das categorias e o classificador.

O ajuste ocorrerá exclusivamente no treinamento. A validação será utilizada para calcular as métricas, sem reajustar o pré-processamento.

A configuração inicial utilizará C = 1,0, solver lbfgs e limite de 2.000 iterações, sem ponderação de classes ou reamostragem.

O limiar será mantido em 0,50 para a comparação inicial. PageValues continuará excluída.

O pipeline completo, os parâmetros, as métricas e a matriz de confusão serão salvos no MLflow.

In [11]:
# ============================================================
# IMPORTAÇÃO DOS COMPONENTES DO MODELO
# ============================================================

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


# ============================================================
# CONSTRUÇÃO DO PIPELINE
# ============================================================

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(logistic_preprocessor),
        ),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                solver="lbfgs",
                max_iter=2000,
                class_weight=None,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)


# ============================================================
# TREINAMENTO E AVALIAÇÃO NA VALIDAÇÃO
# ============================================================

with mlflow.start_run(run_name="logistic_regression_initial") as run:
    logistic_pipeline.fit(X_train, y_train)

    positive_class_index = list(
        logistic_pipeline.classes_
    ).index(1)

    logistic_probabilities = logistic_pipeline.predict_proba(
        X_validation
    )[:, positive_class_index]

    logistic_predictions = (
        logistic_probabilities >= INITIAL_THRESHOLD
    ).astype(int)

    logistic_metrics = {
        "validation_average_precision": average_precision_score(
            y_validation,
            logistic_probabilities,
        ),
        "validation_accuracy": accuracy_score(
            y_validation,
            logistic_predictions,
        ),
        "validation_precision": precision_score(
            y_validation,
            logistic_predictions,
            zero_division=0,
        ),
        "validation_recall": recall_score(
            y_validation,
            logistic_predictions,
            zero_division=0,
        ),
        "validation_f1": f1_score(
            y_validation,
            logistic_predictions,
            zero_division=0,
        ),
    }

    logistic_confusion_matrix = confusion_matrix(
        y_validation,
        logistic_predictions,
        labels=[0, 1],
    )


    # ========================================================
    # REGISTRO DOS PARÂMETROS E DAS MÉTRICAS
    # ========================================================

    classifier = logistic_pipeline.named_steps["classifier"]

    mlflow.log_params(classifier.get_params(deep=False))

    mlflow.log_params(
        {
            "model_type": "LogisticRegression",
            "numeric_preprocessing": "StandardScaler",
            "categorical_preprocessing": "OneHotEncoder",
            "handle_unknown": "ignore",
            "threshold": INITIAL_THRESHOLD,
            "train_rows": len(X_train),
            "validation_rows": len(X_validation),
            "feature_count": X_train.shape[1],
            "excluded_features": ",".join(EXCLUDED_COLUMNS),
        }
    )

    mlflow.set_tags(
        {
            "role": "candidate",
            "evaluation_dataset": "validation",
        }
    )

    mlflow.log_metrics(logistic_metrics)

    mlflow.log_dict(
        {
            "numeric_columns": NUMERIC_COLUMNS,
            "categorical_columns": CATEGORICAL_COLUMNS,
            "excluded_columns": EXCLUDED_COLUMNS,
        },
        "configuration/features.json",
    )

    mlflow.log_dict(
        {
            "labels": [0, 1],
            "rows": "actual",
            "columns": "predicted",
            "matrix": logistic_confusion_matrix.tolist(),
        },
        "evaluation/confusion_matrix.json",
    )


    # ========================================================
    # SALVAMENTO DO PIPELINE COMPLETO
    # ========================================================

    logistic_model_info = mlflow.sklearn.log_model(
        sk_model=logistic_pipeline,
        name="logistic_pipeline",
        input_example=X_train.head(5),
    )

    logistic_run_id = run.info.run_id


# ============================================================
# COMPARAÇÃO COM O MODELO DE REFERÊNCIA
# ============================================================

metric_labels = {
    "validation_average_precision": "Average Precision",
    "validation_accuracy": "Acurácia",
    "validation_precision": "Precisão",
    "validation_recall": "Recall",
    "validation_f1": "F1",
}

logistic_comparison = pd.DataFrame(
    {
        "Referência": baseline_metrics,
        "Regressão Logística": logistic_metrics,
    }
).rename(
    index=metric_labels,
).rename_axis("Métrica")

display(logistic_comparison)


# ============================================================
# APRESENTAÇÃO DA MATRIZ DE CONFUSÃO
# ============================================================

logistic_confusion_display = pd.DataFrame(
    logistic_confusion_matrix,
    index=["Sem compra", "Com compra"],
    columns=["Sem compra", "Com compra"],
).rename_axis(
    index="Resultado real",
    columns="Resultado previsto",
)

display(logistic_confusion_display)

print(f"Identificador da execução: {logistic_run_id}")
print(f"Pipeline salvo no MLflow: {logistic_model_info.model_uri}")
print(
    "Iterações utilizadas: "
    f"{classifier.n_iter_.max()} de {classifier.max_iter}"
)
print("O conjunto de teste permanece reservado.")

/Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer colum

,Referência,Regressão Logística
Métrica,,
Average Precision,0.1547,0.3200
Acurácia,0.8453,0.8442
Precisão,0.0000,0.4583
Recall,0.0000,0.0385
F1,0.0000,0.0710


Resultado previsto,Sem compra,Com compra
Resultado real,,
Sem compra,1550,13
Com compra,275,11


Identificador da execução: e8622391ae124ff2a7d5226d6ebbd38f
Pipeline salvo no MLflow: models:/m-a51f45b7bb6f48d695a9583330b7a466
Iterações utilizadas: 60 de 2000
O conjunto de teste permanece reservado.


#### Resultados da Regressão Logística

A Regressão Logística apresentou Average Precision de aproximadamente 0,3200 na validação, superior ao valor de 0,1547 do modelo de referência.

Esse resultado indica melhor capacidade de ordenar as sessões pela propensão de compra em comparação com a referência, que atribui a mesma probabilidade a todas.

Com o limiar de 0,50, o modelo identificou 11 das 286 sessões com compra, correspondendo a recall de 3,85%.

Foram classificadas 24 sessões como compra: 11 corretamente e 13 incorretamente. A precisão foi de aproximadamente 45,83%, e o F1 foi de 0,0710.

A matriz de confusão apresenta:

- 1.550 sessões sem compra classificadas corretamente.
- 13 sessões sem compra classificadas como compra.
- 275 sessões com compra classificadas como ausência de compra.
- 11 sessões com compra classificadas corretamente.

A acurácia foi de aproximadamente 84,42%, ligeiramente inferior aos 84,53% da referência. Entretanto, a melhora em Average Precision evidencia uma diferença que não é capturada pela acurácia isolada.

O baixo recall mostra que a configuração inicial, com limiar de 0,50, deixa de identificar a maioria das compras. O ajuste do limiar será realizado após a comparação dos modelos, conforme o procedimento definido.

O treinamento utilizou 60 das 2.000 iterações permitidas, sem aviso de convergência na saída apresentada.

O pipeline completo e os resultados foram salvos no MLflow. O conjunto de teste permanece reservado.

---

### 8. Random Forest

A Random Forest será avaliada como alternativa à Regressão Logística, permitindo explorar relações não lineares e interações entre as características.

O pipeline utilizará as variáveis numéricas na escala original e One-Hot Encoding nas variáveis categóricas.

A configuração inicial terá 300 árvores, profundidade máxima de 12 e mínimo de cinco registros por folha. Esses parâmetros são um ponto de partida, não uma configuração comprovadamente ótima.

Não serão utilizadas ponderação de classes ou reamostragem. O limiar inicial permanecerá em 0,50.

O treinamento e o ajuste do pré-processamento utilizarão exclusivamente o conjunto de treinamento.

As métricas serão calculadas na validação e registradas no MLflow, juntamente com os parâmetros, a matriz de confusão e o pipeline completo.

PageValues continuará excluída, e o teste permanecerá reservado.

In [12]:
# ============================================================
# IMPORTAÇÃO DO CLASSIFICADOR
# ============================================================

from sklearn.ensemble import RandomForestClassifier

# ============================================================
# CONSTRUÇÃO DO PIPELINE
# ============================================================

forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(forest_preprocessor),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=5,
                max_features="sqrt",
                class_weight=None,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)


# ============================================================
# TREINAMENTO E AVALIAÇÃO NA VALIDAÇÃO
# ============================================================

with mlflow.start_run(run_name="random_forest_initial") as run:
    forest_pipeline.fit(X_train, y_train)

    positive_class_index = list(
        forest_pipeline.classes_
    ).index(1)

    forest_probabilities = forest_pipeline.predict_proba(
        X_validation
    )[:, positive_class_index]

    forest_predictions = (
        forest_probabilities >= INITIAL_THRESHOLD
    ).astype(int)

    forest_metrics = {
        "validation_average_precision": average_precision_score(
            y_validation,
            forest_probabilities,
        ),
        "validation_accuracy": accuracy_score(
            y_validation,
            forest_predictions,
        ),
        "validation_precision": precision_score(
            y_validation,
            forest_predictions,
            zero_division=0,
        ),
        "validation_recall": recall_score(
            y_validation,
            forest_predictions,
            zero_division=0,
        ),
        "validation_f1": f1_score(
            y_validation,
            forest_predictions,
            zero_division=0,
        ),
    }

    forest_confusion_matrix = confusion_matrix(
        y_validation,
        forest_predictions,
        labels=[0, 1],
    )


    # ========================================================
    # REGISTRO DOS PARÂMETROS E DAS MÉTRICAS
    # ========================================================

    forest_classifier = forest_pipeline.named_steps["classifier"]

    mlflow.log_params(
        forest_classifier.get_params(deep=False)
    )

    mlflow.log_params(
        {
            "model_type": "RandomForestClassifier",
            "numeric_preprocessing": "passthrough",
            "categorical_preprocessing": "OneHotEncoder",
            "handle_unknown": "ignore",
            "threshold": INITIAL_THRESHOLD,
            "train_rows": len(X_train),
            "validation_rows": len(X_validation),
            "feature_count": X_train.shape[1],
            "excluded_features": ",".join(EXCLUDED_COLUMNS),
        }
    )

    mlflow.set_tags(
        {
            "role": "candidate",
            "evaluation_dataset": "validation",
        }
    )

    mlflow.log_metrics(forest_metrics)

    mlflow.log_dict(
        {
            "numeric_columns": NUMERIC_COLUMNS,
            "categorical_columns": CATEGORICAL_COLUMNS,
            "excluded_columns": EXCLUDED_COLUMNS,
        },
        "configuration/features.json",
    )

    mlflow.log_dict(
        {
            "labels": [0, 1],
            "rows": "actual",
            "columns": "predicted",
            "matrix": forest_confusion_matrix.tolist(),
        },
        "evaluation/confusion_matrix.json",
    )


    # ========================================================
    # SALVAMENTO DO PIPELINE COMPLETO
    # ========================================================

    forest_model_info = mlflow.sklearn.log_model(
        sk_model=forest_pipeline,
        name="forest_pipeline",
        input_example=X_train.head(5),
    )

    forest_run_id = run.info.run_id


# ============================================================
# COMPARAÇÃO DOS MODELOS NA VALIDAÇÃO
# ============================================================

model_comparison = pd.DataFrame(
    {
        "Referência": baseline_metrics,
        "Regressão Logística": logistic_metrics,
        "Random Forest": forest_metrics,
    }
).rename(
    index=metric_labels,
).rename_axis("Métrica")

display(model_comparison)


# ============================================================
# APRESENTAÇÃO DA MATRIZ DE CONFUSÃO
# ============================================================

forest_confusion_display = pd.DataFrame(
    forest_confusion_matrix,
    index=["Sem compra", "Com compra"],
    columns=["Sem compra", "Com compra"],
).rename_axis(
    index="Resultado real",
    columns="Resultado previsto",
)

display(forest_confusion_display)

print(f"Identificador da execução: {forest_run_id}")
print(f"Pipeline salvo no MLflow: {forest_model_info.model_uri}")
print("O conjunto de teste permanece reservado.")

/Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/biancafirmino/Developer/ecommerce-purchase-propensity-mlops/.venv/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer colum

,Referência,Regressão Logística,Random Forest
Métrica,,,
Average Precision,0.1547,0.3200,0.3493
Acurácia,0.8453,0.8442,0.8480
Precisão,0.0000,0.4583,1.0000
Recall,0.0000,0.0385,0.0175
F1,0.0000,0.0710,0.0344


Resultado previsto,Sem compra,Com compra
Resultado real,,
Sem compra,1563,0
Com compra,281,5


Identificador da execução: 6934151de0274511909c02ca0808d629
Pipeline salvo no MLflow: models:/m-d53800fb1d8e428a94f866c1b9123d27
O conjunto de teste permanece reservado.


#### Resultados da Random Forest

A Random Forest apresentou Average Precision de aproximadamente 0,3493 na validação, superior aos resultados da Regressão Logística (0,3200) e do modelo de referência (0,1547).

Pelo critério principal definido, essa configuração apresentou o melhor resultado entre os modelos avaliados. A diferença observada não comprova superioridade estatística nem garante o mesmo comportamento em novos dados.

Com limiar de 0,50, foram identificadas cinco das 286 sessões com compra, correspondendo a recall de aproximadamente 1,75%.

As cinco previsões de compra estavam corretas, resultando em precisão de 100% nessa amostra. Entretanto, esse valor se baseia em poucas previsões positivas e deve ser interpretado junto ao baixo recall.

A matriz de confusão apresenta:

- 1.563 sessões sem compra classificadas corretamente.
- Nenhuma sessão sem compra classificada como compra.
- 281 sessões com compra classificadas como ausência de compra.
- Cinco sessões com compra classificadas corretamente.

A acurácia foi de aproximadamente 84,80%, e o F1 foi de 0,0344. Apesar da maior Average Precision, o F1 e o recall foram inferiores aos da Regressão Logística no limiar inicial.

Esses resultados mostram que a ordenação das probabilidades e a classificação em um limiar específico avaliam aspectos diferentes do desempenho.

A Random Forest será selecionada para a próxima etapa, conforme o critério de maior Average Precision na validação. Seu limiar será ajustado pelo F1, conforme o procedimento
estabelecido antes do treinamento.

O pipeline completo e os resultados foram salvos no MLflow. O conjunto de teste permanece reservado.

---

### 9. Seleção do modelo e ajuste do limiar

O modelo será selecionado pela maior Average Precision na validação entre as duas configurações candidatas avaliadas.

Após a seleção, serão avaliados os limiares correspondentes às probabilidades distintas produzidas pelo modelo na validação.

Será escolhido o limiar com maior F1. Em caso de empate, será utilizado o maior limiar.

Uma sessão será classificada como compra quando sua probabilidade for maior ou igual ao limiar escolhido.

O ajuste não modifica o modelo nem as probabilidades. Ele modifica a regra utilizada para converter probabilidades em classes.

Como a validação será utilizada para selecionar o modelo e o limiar, os resultados obtidos nesse conjunto são resultados de seleção, não uma avaliação independente de generalização.

O modelo ajustado no treinamento será mantido sem novo ajuste. A configuração escolhida será avaliada posteriormente no teste.

In [13]:
# ============================================================
# SELEÇÃO DO MODELO PELA AVERAGE PRECISION
# ============================================================

candidates = {
    "Regressão Logística": {
        "pipeline": logistic_pipeline,
        "probabilities": logistic_probabilities,
        "metrics": logistic_metrics,
        "run_id": logistic_run_id,
        "model_uri": logistic_model_info.model_uri,
    },
    "Random Forest": {
        "pipeline": forest_pipeline,
        "probabilities": forest_probabilities,
        "metrics": forest_metrics,
        "run_id": forest_run_id,
        "model_uri": forest_model_info.model_uri,
    },
}

selected_model_name = max(
    candidates,
    key=lambda name: candidates[name]["metrics"][
        f"validation_{PRIMARY_METRIC}"
    ],
)

selected_candidate = candidates[selected_model_name]
selected_pipeline = selected_candidate["pipeline"]
selected_probabilities = selected_candidate["probabilities"]


# ============================================================
# AVALIAÇÃO DOS LIMIARES NA VALIDAÇÃO
# ============================================================

threshold_results = []

for threshold in np.unique(selected_probabilities):
    predictions = (
        selected_probabilities >= threshold
    ).astype(int)

    threshold_results.append(
        {
            "threshold": float(threshold),
            "precision": precision_score(
                y_validation,
                predictions,
                zero_division=0,
            ),
            "recall": recall_score(
                y_validation,
                predictions,
                zero_division=0,
            ),
            "f1": f1_score(
                y_validation,
                predictions,
                zero_division=0,
            ),
        }
    )

threshold_summary = pd.DataFrame(threshold_results)

threshold_ranking = threshold_summary.sort_values(
    ["f1", "threshold"],
    ascending=[False, False],
    kind="stable",
).reset_index(drop=True)

SELECTED_THRESHOLD = float(
    threshold_ranking.loc[0, "threshold"]
)

selected_predictions = (
    selected_probabilities >= SELECTED_THRESHOLD
).astype(int)


# ============================================================
# MÉTRICAS COM O LIMIAR SELECIONADO
# ============================================================

selected_metrics = {
    "validation_average_precision": average_precision_score(
        y_validation,
        selected_probabilities,
    ),
    "validation_accuracy": accuracy_score(
        y_validation,
        selected_predictions,
    ),
    "validation_precision": precision_score(
        y_validation,
        selected_predictions,
        zero_division=0,
    ),
    "validation_recall": recall_score(
        y_validation,
        selected_predictions,
        zero_division=0,
    ),
    "validation_f1": f1_score(
        y_validation,
        selected_predictions,
        zero_division=0,
    ),
}

selected_confusion_matrix = confusion_matrix(
    y_validation,
    selected_predictions,
    labels=[0, 1],
)


# ============================================================
# REGISTRO DA DECISÃO NO MLFLOW
# ============================================================

selection_configuration = {
    "selected_model": selected_model_name,
    "source_run_id": selected_candidate["run_id"],
    "source_model_uri": selected_candidate["model_uri"],
    "model_selection_metric": PRIMARY_METRIC,
    "threshold_selection_metric": THRESHOLD_SELECTION_METRIC,
    "initial_threshold": INITIAL_THRESHOLD,
    "selected_threshold": SELECTED_THRESHOLD,
    "threshold_tie_break": "highest_threshold",
    "prediction_rule": "probability >= selected_threshold",
    "refit": False,
}

with mlflow.start_run(run_name="model_and_threshold_selection") as run:
    mlflow.log_params(selection_configuration)

    mlflow.set_tags(
        {
            "role": "selection",
            "evaluation_dataset": "validation",
        }
    )

    mlflow.log_metrics(selected_metrics)

    mlflow.log_dict(
        selection_configuration,
        "configuration/selection.json",
    )

    mlflow.log_dict(
        {"thresholds": threshold_summary.to_dict(orient="records")},
        "evaluation/threshold_search.json",
    )

    mlflow.log_dict(
        {
            "labels": [0, 1],
            "rows": "actual",
            "columns": "predicted",
            "matrix": selected_confusion_matrix.tolist(),
        },
        "evaluation/confusion_matrix.json",
    )

    selection_run_id = run.info.run_id


# ============================================================
# COMPARAÇÃO ANTES E DEPOIS DO AJUSTE
# ============================================================

threshold_comparison = pd.DataFrame(
    {
        "Limiar inicial": selected_candidate["metrics"],
        "Limiar selecionado": selected_metrics,
    }
).rename(
    index=metric_labels,
).rename_axis("Métrica")

display(threshold_comparison)

display(
    pd.DataFrame(
        selected_confusion_matrix,
        index=["Sem compra", "Com compra"],
        columns=["Sem compra", "Com compra"],
    ).rename_axis(
        index="Resultado real",
        columns="Resultado previsto",
    )
)

print(f"Modelo selecionado: {selected_model_name}")
print(f"Limiar inicial: {INITIAL_THRESHOLD:.2f}")
print(f"Limiar selecionado: {SELECTED_THRESHOLD:.6f}")
print(f"Identificador da seleção: {selection_run_id}")
print("O modelo não foi treinado novamente.")
print("O conjunto de teste permanece reservado.")

,Limiar inicial,Limiar selecionado
Métrica,,
Average Precision,0.3493,0.3493
Acurácia,0.8480,0.6836
Precisão,1.0000,0.2879
Recall,0.0175,0.7098
F1,0.0344,0.4097


Resultado previsto,Sem compra,Com compra
Resultado real,,
Sem compra,1061,502
Com compra,83,203


Modelo selecionado: Random Forest
Limiar inicial: 0.50
Limiar selecionado: 0.180570
Identificador da seleção: ab78182c13da441280d88b67808dbdcc
O modelo não foi treinado novamente.
O conjunto de teste permanece reservado.


#### Resultados da seleção do modelo e do limiar

A Random Forest foi selecionada por apresentar a maior Average Precision na validação entre as configurações candidatas avaliadas.

O limiar escolhido pelo maior F1 foi de aproximadamente 0,180570, inferior ao limiar inicial de 0,50. O valor completo foi preservado no código e no MLflow.

Com esse ajuste, o recall passou de aproximadamente 1,75% para 70,98%, e o F1 aumentou de 0,0344 para 0,4097.

A precisão diminuiu de 100% para aproximadamente 28,79%. Das 705 sessões classificadas como compra, 203 resultaram em compra e 502 não resultaram.

A matriz de confusão apresenta:

- 1.061 sessões sem compra classificadas corretamente.
- 502 sessões sem compra classificadas como compra.
- 83 sessões com compra classificadas como ausência de compra.
- 203 sessões com compra classificadas corretamente.

A acurácia diminuiu de aproximadamente 84,80% para 68,36%. O ajuste ampliou a identificação de compras, acompanhado de um aumento dos falsos positivos.

Average Precision permaneceu em 0,3493, pois o modelo e suas probabilidades não foram modificados.

O limiar maximiza o F1 na validação entre os valores avaliados, mas não representa necessariamente a melhor decisão de negócio. Sua adequação depende dos custos e benefícios das ações associadas às previsões.

Como a validação foi utilizada para selecionar o modelo e o limiar, esses resultados não constituem uma avaliação independente de generalização.

A decisão foi registrada no MLflow, sem novo treinamento. O conjunto de teste permanece reservado para a avaliação final da configuração escolhida.

---

### 10. Avaliação final no conjunto de teste

O conjunto de teste será utilizado para avaliar a configuração selecionada nas etapas anteriores.

Serão mantidos o pipeline ajustado no treinamento e o limiar escolhido na validação, sem novo treinamento ou ajuste.

Serão calculadas Average Precision, acurácia, precisão, recall, F1 e matriz de confusão.

Os resultados serão registrados no MLflow e apresentados ao lado dos resultados da validação.

A validação participou da seleção do modelo e do limiar. Portanto, sua comparação com o teste deve considerar essa diferença de finalidade.

O teste não será utilizado para escolher outro modelo, modificar características ou reajustar o limiar.

A sobreposição de características entre conjuntos, documentada na análise exploratória, permanece como uma limitação desta avaliação.

In [14]:
# ============================================================
# CARREGAMENTO DO CONJUNTO DE TESTE
# ============================================================

TEST_DATASET_PATH = (
    PROCESSED_DATA_DIRECTORY / "test.parquet"
)

if not TEST_DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"Conjunto de teste não encontrado: {TEST_DATASET_PATH}"
    )

df_test = pd.read_parquet(TEST_DATASET_PATH)


# ============================================================
# VERIFICAÇÃO DA ESTRUTURA E DA VARIÁVEL-ALVO
# ============================================================

if df_test.empty:
    raise ValueError("O conjunto de teste está vazio.")

if set(df_test.columns) != set(df_train.columns):
    raise ValueError(
        "O teste possui colunas diferentes das do treinamento."
    )

if df_test[TARGET_COLUMN].isna().any():
    raise ValueError(
        "Existem valores ausentes na variável-alvo do teste."
    )

if not df_test[TARGET_COLUMN].isin([False, True]).all():
    raise ValueError(
        "A variável-alvo do teste deve conter "
        "somente valores booleanos ou códigos 0 e 1."
    )

if df_test[TARGET_COLUMN].nunique() != 2:
    raise ValueError(
        "O conjunto de teste deve conter as duas classes."
    )

X_test = df_test[FEATURE_COLUMNS].copy()
y_test = df_test[TARGET_COLUMN].astype(int).copy()


# ============================================================
# PREDIÇÃO COM O MODELO E O LIMIAR FIXADOS
# ============================================================

positive_class_index = list(
    selected_pipeline.classes_
).index(1)

test_probabilities = selected_pipeline.predict_proba(
    X_test
)[:, positive_class_index]

test_predictions = (
    test_probabilities >= SELECTED_THRESHOLD
).astype(int)


# ============================================================
# CÁLCULO DAS MÉTRICAS NO TESTE
# ============================================================

test_metrics = {
    "test_average_precision": average_precision_score(
        y_test,
        test_probabilities,
    ),
    "test_accuracy": accuracy_score(
        y_test,
        test_predictions,
    ),
    "test_precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "test_recall": recall_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "test_f1": f1_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
}

test_confusion_matrix = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1],
)


# ============================================================
# REGISTRO DA AVALIAÇÃO FINAL NO MLFLOW
# ============================================================

with mlflow.start_run(run_name="final_test_evaluation") as run:
    mlflow.log_params(
        {
            "selected_model": selected_model_name,
            "source_run_id": selected_candidate["run_id"],
            "source_model_uri": selected_candidate["model_uri"],
            "selection_run_id": selection_run_id,
            "threshold": SELECTED_THRESHOLD,
            "test_rows": len(X_test),
            "feature_count": X_test.shape[1],
            "excluded_features": ",".join(EXCLUDED_COLUMNS),
            "refit": False,
        }
    )

    mlflow.set_tags(
        {
            "role": "final_evaluation",
            "evaluation_dataset": "test",
        }
    )

    mlflow.log_metrics(test_metrics)

    mlflow.log_dict(
        {
            "labels": [0, 1],
            "rows": "actual",
            "columns": "predicted",
            "matrix": test_confusion_matrix.tolist(),
        },
        "evaluation/test_confusion_matrix.json",
    )

    test_run_id = run.info.run_id


# ============================================================
# COMPARAÇÃO ENTRE VALIDAÇÃO E TESTE
# ============================================================

evaluation_order = [
    "average_precision",
    "accuracy",
    "precision",
    "recall",
    "f1",
]

evaluation_labels = [
    "Average Precision",
    "Acurácia",
    "Precisão",
    "Recall",
    "F1",
]

final_comparison = pd.DataFrame(
    {
        "Métrica": evaluation_labels,
        "Validação — seleção": [
            selected_metrics[f"validation_{metric}"]
            for metric in evaluation_order
        ],
        "Teste — avaliação final": [
            test_metrics[f"test_{metric}"]
            for metric in evaluation_order
        ],
    }
).set_index("Métrica")

display(final_comparison)


# ============================================================
# MATRIZ DE CONFUSÃO DO TESTE
# ============================================================

test_confusion_display = pd.DataFrame(
    test_confusion_matrix,
    index=["Sem compra", "Com compra"],
    columns=["Sem compra", "Com compra"],
).rename_axis(
    index="Resultado real",
    columns="Resultado previsto",
)

display(test_confusion_display)

print(f"Modelo avaliado: {selected_model_name}")
print(f"Limiar mantido: {SELECTED_THRESHOLD:.6f}")
print(
    "Quantidade de sessões no teste: "
    + f"{len(X_test):,}".replace(",", ".")
)
print(f"Identificador da avaliação: {test_run_id}")
print("Avaliação concluída sem novo treinamento ou ajuste do limiar.")

,Validação — seleção,Teste — avaliação final
Métrica,,
Average Precision,0.3493,0.3786
Acurácia,0.6836,0.6919
Precisão,0.2879,0.2989
Recall,0.7098,0.7378
F1,0.4097,0.4254


Resultado previsto,Sem compra,Com compra
Resultado real,,
Sem compra,1069,495
Com compra,75,211


Modelo avaliado: Random Forest
Limiar mantido: 0.180570
Quantidade de sessões no teste: 1.850
Identificador da avaliação: 5d9f2332e2fb4a848e71121d73d089b6
Avaliação concluída sem novo treinamento ou ajuste do limiar.


#### Resultados da avaliação final no teste

A Random Forest foi avaliada nas 1.850 sessões do teste, mantendo o pipeline ajustado no treinamento e o limiar selecionado na validação, de aproximadamente 0,180570.

Average Precision foi de 0,3786, e o F1 foi de 0,4254. A precisão foi de aproximadamente 29,89%, o recall foi de 73,78% e a acurácia foi de 69,19%.

A matriz de confusão apresenta:

- 1.069 sessões sem compra classificadas corretamente.
- 495 sessões sem compra classificadas como compra.
- 75 sessões com compra classificadas como ausência de compra.
- 211 sessões com compra classificadas corretamente.

Das 706 sessões classificadas como compra, 211 efetivamente resultaram em compra. O modelo identificou a maioria das compras, acompanhado de um número expressivo de falsos positivos.

As métricas foram ligeiramente superiores às observadas na validação. Essa diferença, isoladamente, não comprova estabilidade ou superioridade estatística.

O resultado indica capacidade de discriminação no teste, mas não demonstra viabilidade financeira ou prontidão para uso em produção. A adequação do limiar depende dos custos e benefícios das ações associadas às previsões.

Permanecem as limitações documentadas: sobreposição de características entre conjuntos, uso de informações agregadas da sessão e ausência de validação do cenário de previsão durante a navegação.

A avaliação foi registrada no MLflow, sem novo treinamento ou ajuste do limiar. Os resultados do teste não serão utilizados para modificar a configuração escolhida.

---

### 11. Registro do modelo selecionado

O pipeline selecionado será registrado no MLflow Model Registry, preservando o pré-processamento e o classificador já avaliados.

A versão será associada ao limiar completo escolhido na validação e aos identificadores das execuções de treinamento, seleção e avaliação final.

O registro não envolve novo treinamento nem alteração da configuração com base no teste.

O limiar será armazenado como metadado da versão. Ele deverá ser aplicado explicitamente às probabilidades: a chamada predict do pipeline mantém a regra padrão do classificador.

O registro organiza e versiona o modelo do experimento, sem representar aprovação para uso em produção.

In [15]:
# ============================================================
# IMPORTAÇÃO DO CLIENTE DO MODEL REGISTRY
# ============================================================

from mlflow import MlflowClient

# ============================================================
# REGISTRO DO PIPELINE SELECIONADO
# ============================================================

REGISTERED_MODEL_NAME = "purchase_propensity"

registry_client = MlflowClient()

registered_version = mlflow.register_model(
    model_uri=selected_candidate["model_uri"],
    name=REGISTERED_MODEL_NAME,
)

# ============================================================
# ASSOCIAÇÃO DO LIMIAR E DA ORIGEM DO MODELO
# ============================================================

version_tags = {
    "selected_model": selected_model_name,
    "decision_threshold": repr(float(SELECTED_THRESHOLD)),
    "prediction_rule": "probability >= decision_threshold",
    "positive_class": "1",
    "training_run_id": selected_candidate["run_id"],
    "selection_run_id": selection_run_id,
    "test_run_id": test_run_id,
    "excluded_features": ",".join(EXCLUDED_COLUMNS),
    "usage": "academic_experiment",
}

for tag_name, tag_value in version_tags.items():
    registry_client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=registered_version.version,
        key=tag_name,
        value=tag_value,
    )

registry_client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=registered_version.version,
    description=(
        "Pipeline selecionado pela Average Precision na validação. "
        "Limiar selecionado pelo F1 na validação. "
        "Avaliado no teste sem novo treinamento. "
        "PageValues excluída. "
        "O limiar deve ser aplicado explicitamente ao predict_proba."
    ),
)

# ============================================================
# RECUPERAÇÃO DO MODELO E DO LIMIAR REGISTRADOS
# ============================================================

registered_model_uri = (
    f"models:/{REGISTERED_MODEL_NAME}/{registered_version.version}"
)

version_details = registry_client.get_model_version(
    name=REGISTERED_MODEL_NAME,
    version=registered_version.version,
)

registered_threshold = float(
    version_details.tags["decision_threshold"]
)

registered_pipeline = mlflow.sklearn.load_model(
    registered_model_uri
)

# ============================================================
# VERIFICAÇÃO DA RECUPERAÇÃO SEM UTILIZAR O TESTE
# ============================================================

verification_sample = X_validation.head(10)

registered_positive_index = list(
    registered_pipeline.classes_
).index(1)

original_positive_index = list(
    selected_pipeline.classes_
).index(1)

reloaded_probabilities = registered_pipeline.predict_proba(
    verification_sample
)[:, registered_positive_index]

original_probabilities = selected_pipeline.predict_proba(
    verification_sample
)[:, original_positive_index]

np.testing.assert_allclose(
    reloaded_probabilities,
    original_probabilities,
    rtol=1e-10,
    atol=1e-12,
)

if registered_threshold != SELECTED_THRESHOLD:
    raise ValueError(
        "O limiar recuperado difere do limiar selecionado."
    )

# ============================================================
# APRESENTAÇÃO DO REGISTRO
# ============================================================

registry_summary = pd.DataFrame(
    {
        "Configuração": [
            "Nome no Registry",
            "Versão",
            "Modelo selecionado",
            "URI da versão",
            "Limiar completo",
            "Verificação da recuperação",
        ],
        "Valor": [
            REGISTERED_MODEL_NAME,
            str(registered_version.version),
            selected_model_name,
            registered_model_uri,
            repr(registered_threshold),
            "Probabilidades equivalentes e limiar preservado",
        ],
    }
)

display(registry_summary)

print("Pipeline registrado e recuperado com sucesso.")
print("Nenhum novo treinamento foi realizado.")

Successfully registered model 'purchase_propensity'.
Created version '1' of model 'purchase_propensity'.


,Configuração,Valor
0,Nome no Registry,purchase_propensity
1,Versão,1
2,Modelo selecionado,Random Forest
3,URI da versão,models:/purchase_propensity/1
4,Limiar completo,0.18057002545820874
5,Verificação da recuperação,Probabilidades equivalentes e limiar preservado


Pipeline registrado e recuperado com sucesso.
Nenhum novo treinamento foi realizado.


#### Resultados do registro do modelo

O pipeline da Random Forest foi registrado no MLflow Model Registry com o nome purchase_propensity, versão 1.

A versão pode ser recuperada pela URI: models:/purchase_propensity/1.

A versão 1 corresponde à execução documentada neste notebook. Uma nova execução da célula de registro criará outra versão.

O limiar de decisão foi preservado com precisão completa no metadado decision_threshold, juntamente com as referências às execuções de treinamento, seleção e avaliação final.

O pipeline foi carregado novamente a partir do Registry. Na amostra de dez sessões da validação, suas probabilidades foram equivalentes às do pipeline original, dentro da tolerância numérica definida.

O limiar recuperado também correspondeu exatamente ao valor selecionado.

Essa verificação confirma a recuperação do pipeline e do limiar no ambiente utilizado. Não representa validação em outro ambiente nem aprovação para produção.

Para reproduzir a classificação avaliada, é necessário aplicar explicitamente o limiar às probabilidades da classe positiva retornadas por predict_proba.

Nenhum novo treinamento foi realizado nesta etapa.

---

### 12. Conclusões da modelagem e próximos passos

#### Escopo do experimento

Foram utilizadas 16 características: nove numéricas e sete categóricas. A variável Revenue foi definida como alvo.

PageValues foi excluída por não ter sua construção e disponibilidade no momento da previsão verificadas. Essa decisão não constitui comprovação de vazamento.

O experimento utiliza características agregadas da sessão. Seus resultados não demonstram capacidade de antecipar compras durante a navegação em um instante específico.


#### Comparação dos modelos
Foram avaliados um modelo de referência, uma Regressão Logística e uma Random Forest.

Na validação, Average Precision foi de aproximadamente:

- Referência: 0,1547.
- Regressão Logística: 0,3200.
- Random Forest: 0,3493.

A Random Forest foi selecionada pelo critério previamente definido de maior Average Precision na validação.

A comparação considerou uma configuração inicial de cada modelo candidato, sem busca de hiperparâmetros. Portanto, não demonstra que a Random Forest seja superior a todas as possíveis configurações dos demais algoritmos.


#### Seleção do limiar
O limiar foi selecionado pelo maior F1 na validação, resultando no valor 0.18057002545820874.

O ajuste ampliou a identificação de compras, acompanhado de um aumento dos falsos positivos.

O critério de F1 representa uma escolha técnica inicial. Não foram definidos custos financeiros ou benefícios associados aos diferentes tipos de acerto e erro.


#### Avaliação final
O pipeline e o limiar foram mantidos fixos na avaliação das 1.850 sessões do teste, sem novo treinamento.

Os resultados foram:

- Average Precision: 0,3786.
- Acurácia: 69,19%.
- Precisão: 29,89%.
- Recall: 73,78%.
- F1: 0,4254.

O modelo identificou 211 das 286 sessões com compra. Também classificou 495 sessões sem compra como compra e deixou de identificar 75 compras.

Esses resultados demonstram capacidade de discriminação no conjunto avaliado, mas não comprovam viabilidade financeira ou prontidão para produção.


#### Rastreamento e registro
Os experimentos foram registrados no MLflow com parâmetros, métricas e artefatos.

O pipeline selecionado foi registrado no Model Registry com o nome purchase_propensity. O limiar e as referências às execuções foram associados à versão registrada.

A recuperação do pipeline foi verificada em dez sessões da validação, com probabilidades equivalentes às do modelo original e preservação exata do limiar.

Para reproduzir a classificação avaliada, o limiar deve ser aplicado explicitamente às probabilidades de compra retornadas por predict_proba.


#### Limitações
A seleção do modelo e do limiar utilizou o mesmo conjunto de validação. Seus resultados representam o processo de seleção, não uma avaliação independente.

A sobreposição de características entre conjuntos, documentada na análise exploratória, permanece como uma possível fonte de otimismo na avaliação.

Não foram realizados validação cruzada, análise de incerteza das métricas ou avaliação temporal.

A calibração das probabilidades não foi avaliada. Os valores estimados não devem ser interpretados como probabilidades perfeitamente calibradas.

A disponibilidade das características durante a navegação deverá ser verificada antes de uma aplicação real.

---